# Individual Income Tax Overview, Dependents, and Filing Status

**Selected Excercises**

McGraw Hill's Taxation of Individuals and Business Entities 2025

Spilker, Ayers, Lewis, Weaver, Barrick, Robinson, Worsham

In [55]:
import os
import sys
import pandas as pd
from IPython.display import display, HTML

mod_path = os.path.abspath(os.path.join(".."))
if mod_path not in sys.path:
    sys.path.append(mod_path)

from stackCalc import RPNEngine
from tax_planning import OrdinaryTax, CapitalGains
from income_tax import OrdinaryIncomeTax, PreferentialIncomeTax, TaxCredits    

In [62]:
# Lets define some functions so we can reuse them later
def income_tax_presentation(gi, adj, agi, fad, ti, cg, otl, cgt, crts, pre):
    """
    gi:   Gross income
    adj:  Adjustments to arrive at agi
    fad:  From agi deductions - adjustments from agi. Greater of standard deduction or itemized. Computed by OrdinaryIncomeTax)
    ti:   Taxable income
    cg:   Capital gains
    otl:  Ordinary tax liability
    cgt:  Capital gains tax
    crts: Tax Credits 
    pre:  Prepayments of income tax
    """
    taxable_ordinary_income = ti - cg
    total_tax = otl + cgt
    tax_due_refund = total_tax + crts + pre
    results = "Tax due" if tax_due_refund >= 0 else "Tax refund"

    df = pd.DataFrame([
            ["Gross income", gi],
            ["Adjustments", adj],
            [""],
            ["Adjusted gross income", agi],
            ["From AGI deductions", fad],
            [""],
            ["Taxable income", ti],
            [""],
            ["Taxable ordinary income", taxable_ordinary_income],
            ["Taxable capital gains", cgt],
            [""],
            ["Ordinary income tax", otl],
            ["Capital gains tax", cgt],
            [""],
            ["Tax before credits", total_tax],
            [""],
            ["Credits", crts],
            ["Prepayemnts", pre],
            [results, tax_due_refund],
        ], columns=["", ""])
    return df

---

Jeremy (unmarried) earned 100,000 in salary and 6,000 in interest income during the year. Jeremy’s employer withheld 10,000 of federal income taxes from Jeremy’s paychecks during the year. Jeremy has one qualifying dependent child (age 14) who lives with him. Jeremy qualifies to file as head of household and has 25,000 in itemized deductions.

In [57]:
status = "HOH"
age = 0                      # Not applicable
salary = 100000              # Ordinary income
interest = 6000              # Ordinary income
adjustments = 0              # Above the line adjustments
itemized_deduction = 25000   # Below the line adjustments
standard_deduction = 24150   # Below the line adjustments - 2026 Standard Deduction
tax_withholdings = 10000     # Tax Prepayments
child_tax_credit = 2200      # CTC - Child Tax Credit 2026


**- a) Determine Jeremy’s tax refund or taxes due.**

In [67]:
gross_income = salary + interest
it = OrdinaryIncomeTax(status, age, gross_income, adjustments, itemized_deduction).calculate_tax()
capital_gains = 0
capital_gains_tax = 0
tax = income_tax_presentation(gross_income,
                              it["adjustments"], 
                              it["adjusted_gross_income"], 
                              it["deduction"], 
                              it["taxable_income"],
                              capital_gains,
                              it["ordinary_tax"],
                              capital_gains_tax,
                              -child_tax_credit,
                              -tax_withholdings)

formatter = lambda x: f"${x:,.0f}" if pd.notna(x) else ""

display(HTML(tax.to_html(index=False, formatters={tax.columns[1]: formatter}))) 

ValueError: Unknown format code 'f' for object of type 'str'

**- b) Assume that in addition to the original facts, Jeremy has a long-term capital gain of 4,000. What is Jeremy’s tax refund or tax due including the tax on the capital gain?**

In [51]:
capital_gains = 4000
cgt = PreferentialIncomeTax(status, age, gross_income, 0, capital_gains).calculate_tax()
capital_gains_tax = cgt[1]
tax = income_tax_presentation(gross_income,
                              it["adjustments"], 
                              it["adjusted_gross_income"], 
                              it["deduction"], 
                              it["taxable_income"],
                              capital_gains,
                              it["ordinary_tax"],
                              capital_gains_tax,
                              child_tax_credit,
                              tax_withholdings)


    Gross income                106,000
    Adjustments                  0
                                -------
    Adjusted gross income       106,000
    From AGI deductions          25,000 
                                -------
    Taxable income               81,000
                                -------
    Taxable ordinary income      77,000
    Taxable capital gains             4,000
                                -------
    Ordinary income tax          10,721
    Capital gains tax             600
                                -------
    Tax before credits           11,321

    Credits                       2,200
    Prepayments                  10,000
                                -------
    Tax refund                   -1,479



- c) Assume the original facts except that Jeremy has only 7,000 in itemized deductions. What is Jeremy’s tax refund or tax due?

600.0
